# Convolutional Autoencoder for Image Reconstruction and Denoising

Implementation of a convolutional encoder–decoder that reconstructs and denoises MNIST handwritten-digit images.

**Environment:** Google Colab / Jupyter Notebook  
**Language:** Python

## 1. Aim
Design and train a convolutional autoencoder for image reconstruction and denoising using the MNIST handwritten-digit dataset.

## 2. Theory
An autoencoder learns to reproduce its input. The **encoder** compresses an image into a lower-dimensional latent representation, and the **decoder** reconstructs an image from that representation. For denoising, a corrupted image is supplied as input while the corresponding clean image is used as the training target.

A convolutional architecture is suitable for images because it learns spatial patterns such as edges and strokes.

## 3. Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, Model

np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow version:", tf.__version__)

## 4. Load and preprocess MNIST

In [ ]:
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

# Normalise pixel values to [0, 1] and add a channel dimension.
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("Training images:", x_train.shape)
print("Test images:", x_test.shape)

## 5. Visualise original images

In [ ]:
plt.figure(figsize=(10, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.axis("off")
plt.suptitle("Original MNIST images")
plt.tight_layout()
plt.show()

## 6. Add random noise
Gaussian noise is added to the images. Values are clipped to remain in the valid range [0, 1].

In [ ]:
noise_factor = 0.35

x_train_noisy = x_train + noise_factor * np.random.normal(
    loc=0.0, scale=1.0, size=x_train.shape
).astype("float32")
x_test_noisy = x_test + noise_factor * np.random.normal(
    loc=0.0, scale=1.0, size=x_test.shape
).astype("float32")

x_train_noisy = np.clip(x_train_noisy, 0.0, 1.0)
x_test_noisy = np.clip(x_test_noisy, 0.0, 1.0)

plt.figure(figsize=(10, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train_noisy[i].squeeze(), cmap="gray")
    plt.axis("off")
plt.suptitle("Noisy training images")
plt.tight_layout()
plt.show()

## 7. Build the convolutional autoencoder
The encoder uses convolution and max-pooling to reduce spatial dimensions. The decoder uses upsampling and convolution to reconstruct the original 28 × 28 image.

In [ ]:
input_image = layers.Input(shape=(28, 28, 1), name="noisy_image")

# Encoder
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(input_image)
x = layers.MaxPooling2D((2, 2), padding="same")(x)
x = layers.Conv2D(16, (3, 3), activation="relu", padding="same")(x)
encoded = layers.MaxPooling2D((2, 2), padding="same", name="latent_representation")(x)

# Decoder
x = layers.Conv2D(16, (3, 3), activation="relu", padding="same")(encoded)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
x = layers.UpSampling2D((2, 2))(x)
decoded = layers.Conv2D(1, (3, 3), activation="sigmoid", padding="same",
                        name="reconstructed_image")(x)

autoencoder = Model(input_image, decoded, name="convolutional_denoising_autoencoder")
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.summary()

## 8. Train the denoising autoencoder
The model receives noisy images and learns to predict clean images.

In [ ]:
history = autoencoder.fit(
    x_train_noisy, x_train,
    epochs=5,
    batch_size=128,
    shuffle=True,
    validation_data=(x_test_noisy, x_test),
    verbose=1
)

## 9. Plot training and validation loss

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("Autoencoder training history")
plt.legend()
plt.grid(True)
plt.show()

## 10. Reconstruct and denoise test images

In [ ]:
reconstructed = autoencoder.predict(x_test_noisy[:10], verbose=0)

plt.figure(figsize=(12, 4))
for i in range(10):
    # Noisy input
    plt.subplot(3, 10, i + 1)
    plt.imshow(x_test_noisy[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("Noisy input")

    # Denoised reconstruction
    plt.subplot(3, 10, i + 11)
    plt.imshow(reconstructed[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("Reconstruction")

    # Original target
    plt.subplot(3, 10, i + 21)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("Original")
plt.suptitle("Denoising autoencoder: input, output, and target")
plt.tight_layout()
plt.show()

## 11. Evaluate reconstruction quality

In [ ]:
test_loss = autoencoder.evaluate(x_test_noisy, x_test, verbose=0)
print(f"Test reconstruction loss (binary cross-entropy): {test_loss:.4f}")

# Mean squared error is an additional intuitive reconstruction metric.
mse_noisy = float(np.mean((x_test_noisy - x_test) ** 2))
mse_reconstructed = float(np.mean((reconstructed - x_test[:10]) ** 2))
print(f"MSE of noisy sample images: {mse_noisy:.4f}")
print(f"MSE of reconstructed sample images: {mse_reconstructed:.4f}")

## 12. Observations
- The encoder learns compact spatial features from the input images.
- The decoder reconstructs digit-like structures from the encoded representation.
- Training with noisy inputs and clean targets teaches the network to suppress some noise.
- Reconstruction quality depends on architecture, noise level, dataset size, and training duration.

## 13. Conclusion
A convolutional autoencoder was designed and trained for image denoising. Noisy MNIST images were used as inputs and clean images as targets. The reconstructed images and loss values provide a way to inspect model performance.

## 14. Further experiments
- Change the noise factor and compare results.
- Increase the number of epochs.
- Compare convolutional and dense autoencoders.
- Try salt-and-pepper noise or blur corruption.
- Compare MSE and structural similarity (SSIM).